# customer_shopping_behavior Business Analysis


1. Loading and cleaning the dataset.



In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('customer_shopping_behavior.csv')

print('Shape:', df.shape)
df.head()


Shape: (3900, 18)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [2]:
# -----------------------------
# Data Cleaning
# -----------------------------

clean_df = df.copy()

# Standardize column names
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('/', '_')
)

# Handle missing ratings
clean_df['Review_Rating'] = clean_df['Review_Rating'].fillna(
    clean_df['Review_Rating'].median()
)

# Remove duplicates if any
clean_df = clean_df.drop_duplicates()

# Convert Yes/No style columns to consistent format
for col in ['Subscription_Status', 'Discount_Applied', 'Promo_Code_Used']:
    clean_df[col] = clean_df[col].astype(str).str.strip()

print('Cleaned shape:', clean_df.shape)
clean_df.head()


Cleaned shape: (3900, 18)


,Customer_ID,Age,Gender,Item_Purchased,Category,Purchase_Amount_USD,Location,Size,Color,Season,Review_Rating,Subscription_Status,Shipping_Type,Discount_Applied,Promo_Code_Used,Previous_Purchases,Payment_Method,Frequency_of_Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


## Total revenue generated by male vs female customers

In [3]:
q1 = (
    clean_df.groupby('Gender')['Purchase_Amount_USD']
    .sum()
    .reset_index(name='Total_Revenue')
    .sort_values('Total_Revenue', ascending=False)
)
q1


,Gender,Total_Revenue
1,Male,157890
0,Female,75191


## Customers who used a discount and still spent more than the average purchase amount

In [4]:
avg_purchase = clean_df['Purchase_Amount_USD'].mean()

q2 = clean_df[
    (clean_df['Discount_Applied'].str.lower() == 'yes') &
    (clean_df['Purchase_Amount_USD'] > avg_purchase)
]

q2[['Customer_ID','Item_Purchased','Purchase_Amount_USD']].head()


,Customer_ID,Item_Purchased,Purchase_Amount_USD
1,2,Sweater,64
2,3,Jeans,73
3,4,Sandals,90
6,7,Shirt,85
8,9,Coat,97


## Top 5 products with highest average review rating

In [5]:
q3 = (
    clean_df.groupby('Item_Purchased')['Review_Rating']
    .mean()
    .reset_index(name='Average_Rating')
    .sort_values('Average_Rating', ascending=False)
    .head(5)
)
q3


,Item_Purchased,Average_Rating
6,Gloves,3.861429
14,Sandals,3.844375
3,Boots,3.818750
8,Hat,3.801299
19,Skirt,3.785443


## Compare average purchase amounts between Standard and Express shipping

In [6]:
q4 = (
    clean_df[clean_df['Shipping_Type'].isin(['Standard','Express'])]
    .groupby('Shipping_Type')['Purchase_Amount_USD']
    .mean()
    .reset_index(name='Average_Purchase')
)
q4


,Shipping_Type,Average_Purchase
0,Express,60.475232
1,Standard,58.460245


## Compare average spend and total revenue between subscribers and non-subscribers

In [7]:
q5 = (
    clean_df.groupby('Subscription_Status')
    .agg(
        Average_Spend=('Purchase_Amount_USD','mean'),
        Total_Revenue=('Purchase_Amount_USD','sum')
    )
    .reset_index()
)
q5


,Subscription_Status,Average_Spend,Total_Revenue
0,No,59.865121,170436
1,Yes,59.491928,62645


## Five products with highest percentage of purchases using discounts

In [8]:
discount_flag = (
    clean_df['Discount_Applied']
    .str.lower()
    .eq('yes')
    .astype(int)
)

temp = clean_df.copy()
temp['Discount_Flag'] = discount_flag

q6 = (
    temp.groupby('Item_Purchased')['Discount_Flag']
    .mean()
    .mul(100)
    .reset_index(name='Discount_Percentage')
    .sort_values('Discount_Percentage', ascending=False)
    .head(5)
)

q6


,Item_Purchased,Discount_Percentage
8,Hat,50.000000
20,Sneakers,49.655172
4,Coat,49.068323
23,Sweater,48.170732
13,Pants,47.368421


## Segment customers into New, Returning and Loyal

In [9]:
def customer_segment(x):
    if x == 0:
        return 'New'
    elif x <= 5:
        return 'Returning'
    else:
        return 'Loyal'

customer_segments = (
    clean_df.groupby('Customer_ID')['Previous_Purchases']
    .max()
    .reset_index()
)

customer_segments['Segment'] = customer_segments['Previous_Purchases'].apply(customer_segment)

q7 = customer_segments['Segment'].value_counts().reset_index()
q7.columns = ['Segment','Customer_Count']

q7


,Segment,Customer_Count
0,Loyal,3476
1,Returning,424


## Top 3 most purchased products within each category

In [10]:
product_counts = (
    clean_df.groupby(['Category','Item_Purchased'])
    .size()
    .reset_index(name='Purchase_Count')
)

q8 = (
    product_counts
    .sort_values(['Category','Purchase_Count'], ascending=[True,False])
    .groupby('Category')
    .head(3)
)

q8


,Category,Item_Purchased,Purchase_Count
5,Accessories,Jewelry,171
1,Accessories,Belt,161
7,Accessories,Sunglasses,161
8,Clothing,Blouse,171
12,Clothing,Pants,171
13,Clothing,Shirt,169
20,Footwear,Sandals,160
21,Footwear,Shoes,150
22,Footwear,Sneakers,145
24,Outerwear,Jacket,163


## Are repeat buyers (>5 previous purchases) more likely to subscribe?

In [11]:
repeat_buyers = clean_df[clean_df['Previous_Purchases'] > 5]

q9 = (
    repeat_buyers['Subscription_Status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

q9.columns = ['Subscription_Status','Percentage']
q9


,Subscription_Status,Percentage
0,No,72.439586
1,Yes,27.560414


## Revenue contribution of each age group

In [12]:
bins = [0,18,25,35,45,55,65,100]
labels = ['<18','18-25','26-35','36-45','46-55','56-65','65+']

age_df = clean_df.copy()
age_df['Age_Group'] = pd.cut(age_df['Age'], bins=bins, labels=labels)

q10 = (
    age_df.groupby('Age_Group')['Purchase_Amount_USD']
    .sum()
    .reset_index(name='Revenue')
    .sort_values('Revenue', ascending=False)
)

q10


/tmp/ipykernel_2519/2133132904.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_df.groupby('Age_Group')['Purchase_Amount_USD']


,Age_Group,Revenue
4,46-55,45619
5,56-65,44352
2,26-35,44342
3,36-45,43234
1,18-25,30491
6,65+,20904
0,<18,4139


## Export Clean Data

Use this cleaned dataset for Power BI, Tableau, Excel, or Python visualizations.


In [13]:
clean_df.to_csv('customer_shopping_behavior_cleaned.csv', index=False)

print('Clean dataset exported successfully.')


Clean dataset exported successfully.
